# História dos clusters — Bahia 2024

**Unidade:** município × semana epidemiológica (352 municípios elegíveis × 52 semanas).

**Pergunta:** quais perfis epidemiológicos emergem ao combinar incidência, tendência e dinâmica **entre municípios**?

Escada **v0…v5** (+1 feature por versão). v5 = densidade populacional (hab/km², IBGE Censo 2022).

In [ ]:
from __future__ import annotations

import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", context="notebook")

# Kernel: use o .venv deste repo (Python ia-iv), não o do monorepo pai.
_cwd = Path.cwd().resolve()
if (_cwd / "ml").is_dir():
    ROOT = _cwd
elif (_cwd.parent / "ml").is_dir():
    ROOT = _cwd.parent
else:
    raise RuntimeError(
        "Não achei a pasta ml/. Abra o notebook a partir de ia-iv/ ou ia-iv/notebooks/."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Evita ImportError de símbolos novos quando o kernel já tinha ml carregado.
for name in list(sys.modules):
    if name == "ml" or name.startswith("ml."):
        del sys.modules[name]

from ml.cluster import run_kmeans
from ml.columns import Col, Feat, FEATURE_SETS
from ml.config import DEFAULT_K
from ml.dataset import build_features_panel
from ml.paths import region_manifest_path
from ml.preprocess import build_region_parquet
from ml.config import DEFAULT_YEARS
from ml.experiments import load_experiments
from ml.regions import BA
from ml.validate import transition_matrix

VERSION_BLOCKS = {
    "v0": "baseline: casos",
    "v1": "+ incidência por 100 mil",
    "v2": "+ média móvel (3 semanas)",
    "v3": "+ crescimento semanal",
    "v4": "+ aceleração",
    "v5": "+ densidade populacional (hab/km²)",
}

REGION = BA.slug
ANO = 2024
K = DEFAULT_K
VERSIONS = ["v0", "v1", "v2", "v3", "v4", "v5"]
VERSION_FINAL = "v5"
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00"][:K]
ACCENT = "#0072B2"
SCATTER_KW = dict(s=10, alpha=0.4, linewidths=0.25, edgecolors="#333333")

print(f"Projeto: {ROOT}")
print(f"Python: {sys.executable}")
print(f"Região: {BA.name} | Ano: {ANO} | K={K}")

## Ato 1 — Parquet agregado BA

Um único `dengue.parquet`; população em `populacao.parquet`. Features calculadas em memória.

In [ ]:
build_region_parquet(BA, DEFAULT_YEARS)
manifest = json.loads(region_manifest_path(REGION).read_text(encoding="utf-8"))
pop = pd.read_parquet(ROOT / "data/ml/ba/populacao.parquet")
panel_v0 = build_features_panel(REGION, ANO, "v0")

print(
    f"{manifest['registros_por_ano'].get(str(ANO), '?')} notificações em {ANO} | "
    f"{panel_v0[Col.ID_MUNICIP].nunique()} municípios elegíveis | "
    f"{len(panel_v0):,} linhas (município×semana)"
)
display(pop[["populacao", "area_km2", "densidade_km2"]].describe().round(1))

## Ato 2 — v0: casos absolutos

Separa municípios grandes; baseline limitado (quase ordenação por população).

In [ ]:
res_v0 = run_kmeans(panel_v0, "v0", k=K)
print("silhouette:", round(res_v0.metrics["silhouette"], 3))
display(res_v0.cluster_means.round(2))

## Ato 3 — Escada v0→v5

In [ ]:
def run_version(version: str):
    panel = build_features_panel(REGION, ANO, version)
    result = run_kmeans(panel, version, k=K)
    return {"version": version, "bloco": VERSION_BLOCKS[version], "panel": panel, "result": result}

runs = [run_version(v) for v in VERSIONS]
runs_by = {r["version"]: r for r in runs}

evo = pd.DataFrame([{
    "versão": r["version"],
    "bloco": r["bloco"],
    "silhouette": r["result"].metrics["silhouette"],
    "davies_bouldin": r["result"].metrics["davies_bouldin"],
    "n_feat": len(FEATURE_SETS[r["version"]]),
} for r in runs])
evo["Δ silhouette"] = evo["silhouette"].diff().round(4)
display(evo.round(4))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(evo["versão"], evo["silhouette"], "o-", color=ACCENT, lw=2, markersize=8)
ax.set_ylim(0.82, 0.94)
ax.set_title("Silhouette por versão — BA município×semana")
ax.set_ylabel("silhouette")
plt.tight_layout()

### v1: incidência compara municípios

Salvador sozinha não mostra isso; na Bahia, **v1 ≠ v0**.

In [ ]:
panel_v1 = runs_by["v1"]["panel"]
sem = panel_v1.groupby(Col.SEM_NOT)[Feat.CASOS].sum().idxmax()
snap = panel_v1[panel_v1[Col.SEM_NOT] == sem].nlargest(8, Feat.CASOS)
snap = snap.merge(pop[["id_municip", "municipio"]], left_on=Col.ID_MUNICIP, right_on="id_municip")
display(snap[["municipio", Feat.CASOS, Feat.INCIDENCIA_100K]].round(2))

### Detalhe v4 (dinâmica temporal)

In [ ]:
run_v4 = runs_by["v4"]
panel_v4 = run_v4["panel"]
res_v4 = run_v4["result"]
labeled_v4 = panel_v4.assign(cluster=res_v4.labels)
display(res_v4.cluster_means.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
colors = labeled_v4["cluster"].map({i: PALETTE[i] for i in range(K)})
ax.scatter(
    labeled_v4[Feat.INCIDENCIA_100K],
    labeled_v4[Feat.CRESCIMENTO],
    c=colors,
    **SCATTER_KW,
)
ax.set_xlabel("Incidência / 100 mil")
ax.set_ylabel("Crescimento")
ax.set_title("v4 — incidência × crescimento")
plt.tight_layout()

### v5: densidade populacional

Feature **estática** por município. Silhouette cai levemente vs v4; o ganho é interpretação (urbano vs interior), não métrica interna.

In [ ]:
run_final = runs_by[VERSION_FINAL]
panel_f = run_final["panel"]
res_f = run_final["result"]
labeled_f = panel_f.assign(cluster=res_f.labels)
display(res_f.cluster_means.round(2))

v4_s = runs_by["v4"]["result"].metrics["silhouette"]
v5_s = res_f.metrics["silhouette"]
print(f"silhouette v4={v4_s:.4f} → v5={v5_s:.4f} (Δ {v5_s - v4_s:+.4f})")

urbano = res_f.cluster_means[Feat.DENSIDADE_KM2].idxmax()
print(
    f"Cluster {urbano}: densidade média {res_f.cluster_means.loc[urbano, Feat.DENSIDADE_KM2]:,.0f} hab/km², "
    f"incidência {res_f.cluster_means.loc[urbano, Feat.INCIDENCIA_100K]:.1f} / 100 mil"
)

fig, ax = plt.subplots(figsize=(7, 4))
colors = labeled_f["cluster"].map({i: PALETTE[i] for i in range(K)})
ax.scatter(
    labeled_f[Feat.INCIDENCIA_100K],
    labeled_f[Feat.DENSIDADE_KM2],
    c=colors,
    **SCATTER_KW,
)
ax.set_xlabel("Incidência / 100 mil")
ax.set_ylabel("Densidade (hab/km²)")
ax.set_yscale("log")
ax.set_title("v5 — incidência × densidade")
plt.tight_layout()

## Ato 4 — Log de experimentos

In [ ]:
exp_df = pd.json_normalize(load_experiments())
ba = exp_df[
    (exp_df["config.region"] == REGION)
    & (exp_df["config.year"] == ANO)
    & (exp_df["config.feature_version"].isin(VERSIONS))
].sort_values("id")
display(ba[["id", "tag", "config.feature_version", "metrics.silhouette"]].round(4))

## Ato 5 — Transições (semana t→t+1, por município)

In [ ]:
trans_v0 = transition_matrix(runs_by["v0"]["panel"], runs_by["v0"]["result"].labels)
trans_f = transition_matrix(panel_f, res_f.labels)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, trans, title in [(axes[0], trans_v0, "v0"), (axes[1], trans_f, "v5")]:
    sns.heatmap(trans, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
    ax.set_title(title)
plt.tight_layout()

## Conclusão

1. Cluster = **município×semana** na Bahia (não semanas de uma cidade só).
2. v0 favorece municípios grandes; v1 normaliza por população.
3. v2 a v4 adicionam dinâmica temporal (média móvel, crescimento, aceleração).
4. v5 adiciona densidade: silhouette cai pouco, mas separa perfis urbanos (alta densidade, baixa incidência) de surtos em municípios menores.